# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one pseudonymized content item (page).** Every row also carries a `client_id` (32 distinct clients), but the row itself is per-page, not per-client and not per-day.

**Time window:** all `*_90d` totals cover a single trailing 90-day window ending at the CSV's export time — the same calendar window for every row (there is no `report_date` column, so this is one snapshot, not a time series). Inside that 90 days there are two internal 30-day halves — `*_last_30d` (most recent 30 days) and `*_prev_30d` (the 30 days before that) — used only to build `trend_pct` / `trend_direction`, not a separate row-level window.

Indirect proof the window is a fixed 90+ days: `content_age_days` never goes below 90 in this slice, i.e. only pages old enough to have a complete 90-day history are included here — younger pages are excluded from the slice entirely, not zero-filled.

In [ ]:
import pandas as pd

CSV_URL = (
    "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)
try:
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
except FileNotFoundError:
    df = pd.read_csv(CSV_URL)

print("rows, cols:", df.shape)
print("distinct content_id:", df["content_id"].nunique())
print("distinct client_id:", df["client_id"].nunique())

# Grain check for the unit-of-analysis claim: one row per content item.
dupe_groups = df.groupby("content_id").size()
print("content_id groups with >1 row (should be 0):", (dupe_groups > 1).sum())

# Time-window check: every row should carry a full 90-day history.
print("content_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())
print("age_tier counts:\n", df["age_tier"].value_counts())

# The 90d window splits into two internal 30-day halves used only for the trend fields.
print("\nrows where impressions_prev_30d == 0 (no earlier half to compare to):",
      (df["impressions_prev_30d"] == 0).sum())


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable before we'd decide to touch the page): keyword context (`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`), content properties (`word_count`, `char_count`, `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`), the 90d activity totals (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`), and the derived rates/tiers built from them (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`).
- **Label / proxy** (the target, or what it's computed from — never a feature): `trend_direction`, `trend_pct`, and the two raw columns they're built from, `impressions_last_30d` and `impressions_prev_30d`.
- **Context** (grouping/joining only): `content_id` (row id), `client_id` (32 clients — use for client-holdout splits, never as a feature).
- **Excluded**, each with a why: `provider_used` and `model_used` (operational/production metadata about which LLM wrote the page, not a performance signal — dictionary flags both "not a model feature"); `clicks_last_30d`, `sessions_last_30d`, `clicks_prev_30d`, `sessions_prev_30d` (all four sit inside the exact 30-day halves used to build the label — not literally in the `trend_pct` formula, but one step removed from it, so kept out to avoid near-leakage).

The query cell below confirms two things: that this covers all 44 columns with nothing left unclassified, and that `trend_pct`/`trend_direction` really are computed from `impressions_last_30d` and `impressions_prev_30d` — which is exactly why those four are label/proxy, not features.

In [ ]:
contract = {
    "feature": [
        "search_volume", "competition", "competition_level", "cpc",
        "content_type", "main_intent",
        "word_count", "char_count", "content_age_days", "age_tier", "age_tier_order",
        "days_since_last_update", "freshness_tier", "word_count_tier", "char_count_tier",
        "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
        "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
        "days_with_impressions", "days_with_sessions",
        "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
        "impression_tier", "position_tier",
    ],
    "label_or_proxy": [
        "trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
    ],
    "context": ["content_id", "client_id"],
    "excluded": {
        "provider_used": "LLM vendor that generated the page — an operational/production choice, not a search-performance signal; dictionary marks it explicitly not-a-feature.",
        "model_used": "same reasoning as provider_used, at finer grain (specific model name).",
        "clicks_last_30d": "shares the exact last-30-day window used to build the label; not literally in the trend_pct formula, but one step from it — excluded to avoid near-leakage.",
        "sessions_last_30d": "same last-30-day window as the label — same reasoning.",
        "clicks_prev_30d": "shares the exact prev-30-day window used to build the label — same reasoning.",
        "sessions_prev_30d": "same prev-30-day window as the label — same reasoning.",
    },
}

for bucket, cols in contract.items():
    print(f"{bucket}: {len(cols)} fields")

classified = (set(contract["feature"]) | set(contract["label_or_proxy"])
              | set(contract["context"]) | set(contract["excluded"].keys()))
print("\nevery one of the 44 columns classified?", set(df.columns) == classified,
      "| unclassified:", set(df.columns) - classified)

# Verify the label-trap claim directly: recompute trend_pct from the two raw 30-day
# columns and confirm it matches the shipped column (allowing for rounding).
denom = df["impressions_prev_30d"]
recomputed_pct = ((df["impressions_last_30d"] - denom) / denom * 100).round(1)
mask = denom != 0
match_rate = (recomputed_pct[mask] == df.loc[mask, "trend_pct"]).mean()
print("\ntrend_pct recomputed from impressions_last_30d & impressions_prev_30d, match rate:",
      round(match_rate, 3))
print("-> trend_pct (and trend_direction, and the label built from it) is DERIVED from those",
      "two columns, so none of the four belong in 'feature'.")


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four claims from sections 1–2, each checked below:

1. **Grain** — one row per `content_id`: grouping by `content_id` and counting produces zero groups with more than one row.
2. **Counts** — 30,000 rows across 32 clients, but very unevenly split (rows-per-client ranges from 3 to 7,008, median 567) — a fact section 4 leans on.
3. **Missing values** — missingness is patterned by `content_type`, not random: `word_count` is only missing for "keyword article" rows (28.3%), while `search_volume` is only missing for "feedly article" rows (100%). A blind `fillna(0)` on either would silently encode content type into the feature.
4. **Windows** — `content_age_days` never drops below 90 (confirms the fixed 90-day window from section 1), and `trend_pct` is blank for exactly the 3,388 rows where `impressions_prev_30d == 0` — the blanks are a division-by-zero artifact of the window split, not random missingness. `avg_position == 0` for 1,205 rows, which per the dictionary means "no position data," not rank zero.

In [ ]:
# --- Grain ---
dupe_groups = df.groupby("content_id").size()
print("GRAIN: content_id groups with >1 row:", (dupe_groups > 1).sum(), "(0 = one row per content item, confirmed)")

# --- Counts ---
print("\nCOUNTS: rows =", len(df), "| distinct clients =", df["client_id"].nunique())
per_client = df.groupby("client_id").size()
print("rows per client -> min/median/max:", per_client.min(), per_client.median(), per_client.max())

# --- Missing values: overall, then split by content_type to test "random vs patterned" ---
miss_overall = df.isna().mean().sort_values(ascending=False)
print("\nMISSINGNESS (overall, cols with any NaN):")
print((miss_overall[miss_overall > 0] * 100).round(1))

print("\nMISSINGNESS by content_type (the pattern the dictionary warns about):")
print("word_count missing % by content_type:")
print((df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean() * 100)).round(1))
print("\nsearch_volume missing % by content_type:")
print((df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean() * 100)).round(1))

# --- Windows ---
print("\nWINDOWS: content_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())
print("trend_pct blank count:", df["trend_pct"].isna().sum(),
      "| impressions_prev_30d == 0 count:", (df["impressions_prev_30d"] == 0).sum(),
      "(these two match -> trend_pct is blank exactly when the prev-30d denominator is 0, not at random)")
print("avg_position == 0 count:", (df["avg_position"] == 0).sum(), "(dictionary: this means 'no position data', not rank zero)")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell you, each backed by a number:

1. **Unbalanced clients** — rows per client range 3 to 7,008 (median 567); any pooled statistic is dominated by a few large clients, so a cross-client comparison needs client-level weighting or a client-holdout split, not a naive pool.
2. **Age floor / survivorship** — every row has `content_age_days ≥ 90`; brand-new content (<90 days old) is invisible to this slice, so nothing here describes early-life decay, only decay after a page has already survived 90 days.
3. **Content-type-patterned gaps** — `word_count` missing for 28.3% of "keyword article" rows, `search_volume` missing for 100% of "feedly article" rows: "missing" here means "wrong content_type for this metric," not a random hole, so any model must add `has_*` flags rather than impute.
4. **One snapshot, not a time series** — there is no `report_date`/export-date column; this is a single 90-day cross-section per page. The only "trend" available is the one last-30-vs-prev-30 comparison baked into `trend_pct` — it cannot show whether a page's trajectory is stable across multiple windows.
5. **Mismatched measurement systems** — `ai_traffic_pct` exceeds 100% for 23 rows (AI-referred sessions are counted independently of total GA4 sessions) and `avg_position == 0` for 1,205 rows means "no ranking data," not rank zero; neither column is a clean percentage/rank without that caveat.
6. **No causal design** — `provider_used`/`model_used` are excluded as features (section 2), so this data can show association at best; it cannot support a claim like "provider X causes decline," since no experiment assigned providers to pages.

In [ ]:
# Each limit below is backed by a number pulled straight from the frame.

per_client = df.groupby("client_id").size()
print("1) Unbalanced clients: rows per client range", per_client.min(), "-", per_client.max(),
      "| median", per_client.median(), "-> a handful of clients dominate any pooled statistic.")

print("\n2) Age floor / survivorship: content_age_days min =", df["content_age_days"].min(),
      "-> every row already survived a full 90-day window; brand-new pages (<90 days old) are invisible here.")

kw_missing = df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean() * 100, 1)).to_dict()
sv_missing = df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean() * 100, 1)).to_dict()
print("\n3) Content-type-patterned gaps: word_count missing %", kw_missing,
      "| search_volume missing %", sv_missing,
      "-> 'missing' means 'wrong content_type for this metric', not a random hole.")

print("\n4) One snapshot, not a time series: no report_date/export_date column exists in this CSV",
      "-> this is a single 90-day cross-section per page; the only 'trend' available is the",
      "one last-30-vs-prev-30 comparison baked into trend_pct, not a real multi-period history.")

exceeds_100 = (df["ai_traffic_pct"] > 100).sum()
no_position = (df["avg_position"] == 0).sum()
print("\n5) Mismatched measurement systems: ai_traffic_pct > 100 for", exceeds_100, "rows",
      "(AI-referred sessions are counted by a different system than total GA4 sessions), and",
      "avg_position == 0 for", no_position, "rows means 'no ranking data', not rank zero",
      "-> neither column can be read as a clean percentage/rank without this caveat.")

print("\n6) No causal design: provider_used/model_used are excluded as features (contract section 2)",
      "-> this data can show association at most; it cannot support a claim like",
      "'provider X causes decline', because no experiment assigned providers to pages.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.